# Day 13 — HuggingFace Live Demo
## Transformers & The Attention Revolution

---

> **For the teacher:** Run each cell live. Students only watch.  
> Each section has a short explanation cell followed by a code cell.  
> Pre-run all cells before class to cache model weights — first download can take 2–3 minutes.

---

### What is HuggingFace?

HuggingFace is the open-source home of pretrained AI models — think of it as the GitHub of machine learning. It hosts over **100,000 pretrained models** that anyone can download and use in a few lines of Python.

The key abstraction is the **`pipeline()`** function:

```
Raw Text  →  Tokenizer  →  Pretrained Model  →  Post-processing  →  Output
```

You call `pipeline()` with a task name. It handles everything else — downloading weights, tokenising input, running the model, formatting the output. You don't need to know the model architecture or manage tensors.

**Today's demos:**
1. Sentiment Analysis
2. Named Entity Recognition
3. Question Answering
4. Summarization
5. Zero-Shot Classification

---

## Section 0 — Setup

Install the `transformers` library from HuggingFace and `torch` (PyTorch) as the backend.  
Run this once. If already installed, it skips silently.

In [73]:
# Install dependencies (run once)
# If you are in a pre-configured conda environment, this may already be installed
%pip install transformers torch --quiet

Note: you may need to restart the kernel to use updated packages.


In [74]:
# Core imports
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')  # suppress tokenizer warnings for clean output

print("HuggingFace Transformers loaded successfully.")
print("Ready to run demos.")

HuggingFace Transformers loaded successfully.
Ready to run demos.


---

## Demo 1 — Sentiment Analysis

### What it does
Given a piece of text, the model predicts whether it expresses a **positive** or **negative** sentiment, along with a confidence score.

### Where it's used
- Product review monitoring (Amazon, Daraz)
- Social media brand tracking
- Customer support ticket prioritisation
- Financial news sentiment for trading signals

### What model runs under the hood
By default, `pipeline('sentiment-analysis')` loads **DistilBERT** — a distilled (compressed) version of BERT that is 40% smaller and 60% faster while retaining 97% of BERT's performance. It was fine-tuned on the Stanford Sentiment Treebank (SST-2) dataset.

### Output format
```python
[{'label': 'POSITIVE', 'score': 0.9998}]
```
- `label` — predicted class
- `score` — model's confidence (probability from softmax)

---

> **Ask students before running:** *"If I pass the sentence 'This model is incredible' — what label and score do you expect?"*

In [75]:
# Load the sentiment analysis pipeline
# First run downloads ~250MB of model weights (DistilBERT)
sentiment_pipeline = pipeline("sentiment-analysis")

print("Sentiment pipeline loaded.")
print(f"Model: {sentiment_pipeline.model.config._name_or_path}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11125.19it/s]


Sentiment pipeline loaded.
Model: distilbert/distilbert-base-uncased-finetuned-sst-2-english


In [76]:
# --- Test 1: Clearly positive ---
text_positive = "This model is incredible. I cannot believe how well it understands language."

result = sentiment_pipeline(text_positive)
print(f"Text:   {text_positive}")
print(f"Result: {result}")

Text:   This model is incredible. I cannot believe how well it understands language.
Result: [{'label': 'POSITIVE', 'score': 0.9998052716255188}]


In [77]:
# --- Test 2: Clearly negative ---
text_negative = "The training was too long, poorly explained, and completely wasted my time."

result = sentiment_pipeline(text_negative)
print(f"Text:   {text_negative}")
print(f"Result: {result}")

Text:   The training was too long, poorly explained, and completely wasted my time.
Result: [{'label': 'NEGATIVE', 'score': 0.9998106360435486}]


In [78]:
# --- Test 3: Ambiguous — does the model handle nuance? ---
text_ambiguous = "The food was decent but the service was absolutely terrible."

result = sentiment_pipeline(text_ambiguous)
print(f"Text:   {text_ambiguous}")
print(f"Result: {result}")
print()
print("Notice: the model gives one label for the whole sentence.")
print("The negative service likely outweighs the positive food.")

Text:   The food was decent but the service was absolutely terrible.
Result: [{'label': 'NEGATIVE', 'score': 0.9905843734741211}]

Notice: the model gives one label for the whole sentence.
The negative service likely outweighs the positive food.


In [79]:
# --- Test 4: Nepal context ---
texts_nepal = [
    "Kathmandu traffic has become completely unmanageable. It ruins every morning.",
    "The view of the Himalayas from Nagarkot was breathtaking. Absolutely worth the trip.",
    "Load shedding has improved a lot over the last few years in Nepal."
]

results = sentiment_pipeline(texts_nepal)
print("Batch prediction on multiple sentences:")
print("-" * 70)
for text, result in zip(texts_nepal, results):
    label = result['label']
    score = result['score']
    print(f"[{label:8s} | {score:.3f}]  {text[:60]}..." if len(text) > 60 else f"[{label:8s} | {score:.3f}]  {text}")

Batch prediction on multiple sentences:
----------------------------------------------------------------------
[NEGATIVE | 1.000]  Kathmandu traffic has become completely unmanageable. It rui...
[POSITIVE | 1.000]  The view of the Himalayas from Nagarkot was breathtaking. Ab...
[POSITIVE | 0.999]  Load shedding has improved a lot over the last few years in ...


**Key observations:**
- The model handles batch input (list of sentences) efficiently
- Confidence scores close to 1.0 indicate high certainty
- Ambiguous sentences produce lower confidence scores
- The model was trained on English text but understands domain-specific language (traffic, Himalayas)

---

## Demo 2 — Named Entity Recognition (NER)

### What it does
Identifies and classifies **named entities** in text — people, organisations, locations, dates, and more. This is a token classification task: every token in the input gets a label.

### Entity types (standard CoNLL tags)
| Tag | Meaning | Example |
|-----|---------|--------|
| PER | Person | *Elon Musk*, *Prithvi Narayan Shah* |
| ORG | Organisation | *SpaceX*, *Fusemachines*, *TechAxis* |
| LOC | Location | *Kathmandu*, *Mount Everest* |
| MISC | Miscellaneous | *Nepali*, *English* |

### Where it's used
- News article analysis (extracting who said what, where)
- Legal document processing (identify parties, dates, amounts)
- Medical records (extract drug names, diagnoses, dosages)
- Resume parsing (extract skills, companies, universities)

### `aggregation_strategy="simple"`
Without this, the model returns one label per token (subword). With `simple`, it merges adjacent tokens that form the same entity — so "New" and "York" become one entity "New York".

---

> **Ask students:** *"In the sentence 'Elon Musk founded SpaceX in Hawthorne, California' — how many entities do you expect, and what types?"*

In [80]:
# Load NER pipeline
# Default model: bert-base-NER (fine-tuned BERT on CoNLL-2003 dataset)
ner_pipeline = pipeline("ner", aggregation_strategy="simple")

print("NER pipeline loaded.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9763.25it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER pipeline loaded.


In [81]:
# --- Test 1: Classic example ---
text1 = "Elon Musk founded SpaceX in Hawthorne, California in 2002."

entities = ner_pipeline(text1)
print(f"Input: {text1}")
print()
print(f"{'Entity':<25} {'Type':<8} {'Confidence':<12}")
print("-" * 50)
for e in entities:
    print(f"{e['word']:<25} {e['entity_group']:<8} {e['score']:.3f}")

Input: Elon Musk founded SpaceX in Hawthorne, California in 2002.

Entity                    Type     Confidence  
--------------------------------------------------
Elon Musk                 PER      0.999
SpaceX                    ORG      0.999
Hawthorne                 LOC      0.991
California                LOC      0.998


In [82]:
# --- Test 2: Nepal-specific context ---
text2 = "Fusemachines, an AI company headquartered in Kathmandu, Nepal, was founded by Sameer Maskey."

entities = ner_pipeline(text2)
print(f"Input: {text2}")
print()
print(f"{'Entity':<30} {'Type':<8} {'Confidence':<12}")
print("-" * 55)
for e in entities:
    print(f"{e['word']:<30} {e['entity_group']:<8} {e['score']:.3f}")

Input: Fusemachines, an AI company headquartered in Kathmandu, Nepal, was founded by Sameer Maskey.

Entity                         Type     Confidence  
-------------------------------------------------------
Fusemachines                   ORG      0.995
Kathmandu                      LOC      0.999
Nepal                          LOC      1.000
Sameer Maskey                  PER      0.999


In [83]:
# --- Test 3: News article style ---
text3 = (
    "The World Bank approved a $250 million loan for Nepal on Tuesday. "
    "Finance Minister Bishnu Poudel welcomed the decision in Kathmandu, "
    "stating it would support infrastructure development across the country."
)

entities = ner_pipeline(text3)
print(f"Input:\n{text3}")
print()
print(f"{'Entity':<30} {'Type':<8} {'Confidence':<12}")
print("-" * 55)
for e in entities:
    print(f"{e['word']:<30} {e['entity_group']:<8} {e['score']:.3f}")

Input:
The World Bank approved a $250 million loan for Nepal on Tuesday. Finance Minister Bishnu Poudel welcomed the decision in Kathmandu, stating it would support infrastructure development across the country.

Entity                         Type     Confidence  
-------------------------------------------------------
World Bank                     ORG      0.999
Nepal                          LOC      1.000
Bishnu Poudel                  PER      0.995
Kathmandu                      LOC      0.999


**Key observations:**
- BERT-based NER identifies entities without any rules or dictionaries — purely learned from training data
- Multi-word entities ("World Bank", "Bishnu Poudel") are correctly grouped
- Confidence scores are per-entity — lower scores indicate ambiguous cases
- This is a classic BERT use case: token-level classification requiring bidirectional context

---

## Demo 3 — Extractive Question Answering

### What it does
Given a **context** (a passage of text) and a **question**, the model finds the exact span of text in the context that answers the question. This is called **extractive QA** — the model does not generate a new answer, it locates the answer within the provided text.

### How it works internally
The model takes `[CLS] question [SEP] context [SEP]` as input. It outputs two probability distributions — one for the start position of the answer, one for the end position. The span with the highest combined start+end probability is returned as the answer.

### This is BERT in its element
Because BERT reads both directions simultaneously, it can use the question to attend to relevant parts of the context and vice versa. GPT-style models struggle here because they can only read left to right.

### Where it's used
- Internal document search ("Find the refund policy in our terms of service")
- Medical literature review
- Legal contract analysis
- Customer support FAQ automation

---

> **Ask students:** *"What does the model return — a generated answer, or something extracted directly from the text?"*

In [84]:
# Load QA pipeline
# Default model: deepset/roberta-base-squad2 (fine-tuned on SQuAD 2.0)
qa_pipeline = pipeline("document-question-answering", model="deepset/roberta-base-squad2")

print("Question-Answering pipeline loaded.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11210.65it/s]
[transformers] The model 'RobertaForQuestionAnswering' is not supported for document-question-answering. Supported models are ['PeftModelForQuestionAnswering', 'LayoutLMForQuestionAnswering', 'LayoutLMv2ForQuestionAnswering', 'LayoutLMv3ForQuestionAnswering'].


Question-Answering pipeline loaded.


In [87]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
import torch

model_name = "deepset/roberta-base-squad2"
qa_tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_name)

def qa_pipeline(question, context):
    inputs = qa_tokenizer(question, context, return_tensors="pt")
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start = outputs.start_logits.argmax()
    end = outputs.end_logits.argmax() + 1
    answer = qa_tokenizer.convert_tokens_to_string(
        qa_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start:end])
    )
    import torch.nn.functional as F
    score = float(F.softmax(outputs.start_logits, dim=-1).max())
    return {"answer": answer, "score": score}

print("Question-Answering pipeline loaded.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18298.87it/s]

Question-Answering pipeline loaded.


In [86]:
# --- Context: Nepal geography ---
context_nepal = """
Nepal is a landlocked country in South Asia, located in the Himalayas. 
It is bordered by China to the north and India to the south, east, and west. 
Kathmandu is the capital and largest city of Nepal, with a population of approximately 
1.4 million people. Nepal is home to eight of the world's ten tallest mountains, 
including Mount Everest, which at 8,849 metres is the highest point on Earth. 
The country covers an area of 147,181 square kilometres and has a population 
of approximately 30 million people. Nepali is the official language.
"""

questions = [
    "What is the capital of Nepal?",
    "How tall is Mount Everest?",
    "What is the official language of Nepal?",
    "How many of the world's tallest mountains are in Nepal?"
]

print("Context: Nepal Geography")
print("-" * 70)
for q in questions:
    result = qa_pipeline(question=q, context=context_nepal)
    print(f"Q: {q}")
    print(f"A: {result['answer']}  (confidence: {result['score']:.3f})")
    print()

Context: Nepal Geography
----------------------------------------------------------------------


TypeError: DocumentQuestionAnsweringPipeline.__call__() missing 1 required positional argument: 'image'

In [ ]:
# --- Context: Nepal geography ---
context_nepal = """
Nepal is a landlocked country in South Asia, located in the Himalayas. 
It is bordered by China to the north and India to the south, east, and west. 
Kathmandu is the capital and largest city of Nepal, with a population of approximately 
1.4 million people. Nepal is home to eight of the world's ten tallest mountains, 
including Mount Everest, which at 8,849 metres is the highest point on Earth. 
The country covers an area of 147,181 square kilometres and has a population 
of approximately 30 million people. Nepali is the official language.
"""

questions = [
    "What is the capital of Nepal?",
    "How tall is Mount Everest?",
    "What is the official language of Nepal?",
    "How many of the world's tallest mountains are in Nepal?"
]

print("Context: Nepal Geography")
print("-" * 70)
for q in questions:
    result = qa_pipeline(question=q, context=context_nepal)
    print(f"Q: {q}")
    print(f"A: {result['answer']}  (confidence: {result['score']:.3f})")
    print()

In [ ]:
# --- Context: Transformers (meta-demo!) ---
context_transformers = """
The Transformer architecture was introduced in the paper 'Attention Is All You Need' 
by Vaswani et al. in 2017. Unlike recurrent neural networks, the Transformer relies 
entirely on a self-attention mechanism to draw global dependencies between input and output. 
BERT, published by Google in 2018, uses only the encoder part of the Transformer and 
introduced bidirectional pre-training via masked language modelling. GPT, published by 
OpenAI in 2018, uses only the decoder part and generates text autoregressively by 
predicting the next token. GPT-3, released in 2020, has 175 billion parameters and 
demonstrated remarkable few-shot learning capabilities.
"""

questions_transformers = [
    "Who introduced the Transformer architecture?",
    "When was BERT published?",
    "How many parameters does GPT-3 have?",
    "What pre-training method does BERT use?"
]

print("Context: Transformer History")
print("-" * 70)
for q in questions_transformers:
    result = qa_pipeline(question=q, context=context_transformers)
    print(f"Q: {q}")
    print(f"A: {result['answer']}  (confidence: {result['score']:.3f})")
    print()

Context: Transformer History
----------------------------------------------------------------------


TypeError: DocumentQuestionAnsweringPipeline.__call__() missing 1 required positional argument: 'image'

In [ ]:
# --- What happens when the answer isn't in the context? ---
# SQuAD 2.0 trained models can say 'I don't know' (low score)

result_unanswerable = qa_pipeline(
    question="What is the population of China?",
    context=context_nepal
)
print("Unanswerable question test:")
print(f"Q: What is the population of China?")
print(f"A: {result_unanswerable['answer']}  (confidence: {result_unanswerable['score']:.3f})")
print()
print("Low confidence score indicates the model is uncertain.")
print("In production, you would set a confidence threshold (e.g. 0.2) below which")
print("you return 'Answer not found' instead of a potentially wrong answer.")

**Key observations:**
- The model returns the exact span from the context — it does not paraphrase or generate
- Confidence scores are useful for thresholding in production (reject low-confidence answers)
- This is why bidirectional models like BERT excel at QA: the question and context attend to each other fully
- In production, you would chunk long documents and run QA over each chunk

---

## Demo 4 — Summarization

### What it does
Takes a long piece of text and produces a **shorter version** that captures the main points. This is **abstractive summarization** — unlike extractive QA, the model generates new sentences that were not necessarily in the original text.

### This is generative — note the difference from QA
QA extracts a span. Summarization generates new text. This requires a decoder. The default model is **Facebook BART** (encoder-decoder architecture) fine-tuned on the CNN/DailyMail news summarization dataset.

### Key parameters
- `max_length` — maximum tokens in the summary
- `min_length` — minimum tokens (prevents trivially short summaries)
- `do_sample=False` — use greedy/beam search instead of sampling (more deterministic)

### Where it's used
- News digest applications
- Legal document summaries
- Meeting transcript summarization
- Research paper abstracts

---

> **Ask students:** *"Is the summary copy-pasted from the article, or is it newly written? How is that different from QA?"*

In [ ]:
# Load summarization pipeline
# Default model: facebook/bart-large-cnn
summarizer = pipeline("summarization")

print("Summarization pipeline loaded.")
print(f"Model: {summarizer.model.config._name_or_path}")

In [ ]:
# --- Article 1: AI news style ---
article1 = """
Artificial intelligence has transformed the technology industry over the past decade, 
with large language models now powering products used by hundreds of millions of people 
around the world. Companies like OpenAI, Google, Anthropic, and Meta have invested 
billions of dollars in developing increasingly capable models. These models are trained 
on vast datasets containing text from the internet, books, and scientific papers. 
The training process requires enormous computing infrastructure — clusters of thousands 
of specialised chips called GPUs or TPUs running for months at a time. 
The environmental cost of this training has attracted growing attention, 
with some researchers estimating that training a large model produces as much carbon 
dioxide as five cars over their full lifetimes. Despite these concerns, investment 
in AI continues to accelerate as companies compete for dominance in what many 
consider to be the most transformative technology since the internet.
"""

summary1 = summarizer(article1, max_length=80, min_length=30, do_sample=False)

print("Original article length:", len(article1.split()), "words")
print("Summary length:", len(summary1[0]['summary_text'].split()), "words")
print()
print("SUMMARY:")
print(summary1[0]['summary_text'])

In [ ]:
# --- Article 2: Nepal tech sector ---
article2 = """
Nepal's technology sector has been growing steadily over the past several years, 
driven by a young and increasingly educated workforce, expanding internet connectivity, 
and a growing number of startups focused on fintech, e-commerce, and artificial intelligence. 
Companies such as Fusemachines and Leapfrog Technology have established Nepal as a 
credible destination for software development and AI services. 
The government has announced several digital Nepal initiatives aimed at improving 
digital infrastructure across the country, including efforts to bring reliable 
internet connectivity to remote mountainous regions. 
Despite challenges including political instability, infrastructure gaps, and 
brain drain as skilled workers migrate abroad for better opportunities, 
the sector continues to attract investment and talent. 
Several Nepali engineers have returned from positions at major technology companies 
in the United States and are founding their own startups in Kathmandu, 
contributing to an emerging entrepreneurial ecosystem.
"""

summary2 = summarizer(article2, max_length=80, min_length=30, do_sample=False)

print("Original article length:", len(article2.split()), "words")
print("Summary length:", len(summary2[0]['summary_text'].split()), "words")
print()
print("SUMMARY:")
print(summary2[0]['summary_text'])

In [ ]:
# --- Comparing different length constraints ---
print("Effect of max_length on summary output:")
print("=" * 60)

for max_len in [40, 80, 120]:
    result = summarizer(article1, max_length=max_len, min_length=20, do_sample=False)
    summary_text = result[0]['summary_text']
    print(f"max_length={max_len} ({len(summary_text.split())} words):")
    print(f"  {summary_text}")
    print()

**Key observations:**
- The summary is **generated** — sentences that may not appear verbatim in the source
- This is fundamentally different from QA (which extracts spans) — it requires a generative decoder
- `max_length` and `min_length` control the compression ratio
- BART's encoder-decoder architecture is ideal here: encoder reads and understands the full article, decoder generates a fluent summary
- The model retains the key facts but rephrases them — this is abstractive summarization

---

## Demo 5 — Zero-Shot Classification

### What it does
Classifies text into categories **without any task-specific training data**. You provide the text and a list of candidate labels. The model figures out which label best fits — even if it has never seen those exact labels before.

### How it works
The model (typically BART fine-tuned on Natural Language Inference) treats classification as a textual entailment problem:
- *"Does this text entail: 'This example is about LABEL'?"*
- It scores each label using entailment probability
- The label with the highest entailment score wins

### Why this is remarkable
Traditional ML classifiers require labelled training data for each class. Zero-shot classification works with **zero labelled examples**. You can change the labels at runtime without retraining the model.

### Where it's used
- Content moderation (classify new types of harmful content without retraining)
- Document routing (send this email to HR / Finance / IT)
- News article categorization
- Rapid prototyping before collecting labelled data

---

> **Ask students:** *"How many labelled training examples does this model need to classify into our custom categories?"* (Answer: zero — that's why it's called zero-shot.)

In [ ]:
# Load zero-shot classification pipeline
# Model: facebook/bart-large-mnli (BART fine-tuned on Multi-Genre NLI dataset)
zeroshot_pipeline = pipeline("zero-shot-classification")

print("Zero-shot classification pipeline loaded.")

In [ ]:
# --- Test 1: Topic classification ---
text = "The stock market crashed by 5% today following the central bank's interest rate announcement."
candidate_labels = ["finance", "sports", "politics", "technology", "entertainment"]

result = zeroshot_pipeline(text, candidate_labels)

print(f"Text: {text}")
print(f"Candidate labels: {candidate_labels}")
print()
print(f"{'Label':<15} {'Score':<10}")
print("-" * 28)
for label, score in zip(result['labels'], result['scores']):
    bar = '█' * int(score * 30)
    print(f"{label:<15} {score:.3f}  {bar}")

In [ ]:
# --- Test 2: Sentiment with custom labels (not positive/negative) ---
review = "The hotel room was spacious and clean, but the breakfast was disappointing and overpriced."
custom_labels = ["would recommend", "would not recommend", "neutral"]

result = zeroshot_pipeline(review, custom_labels)

print(f"Text: {review}")
print()
print(f"{'Label':<22} {'Score':<10}")
print("-" * 35)
for label, score in zip(result['labels'], result['scores']):
    bar = '█' * int(score * 30)
    print(f"{label:<22} {score:.3f}  {bar}")

In [ ]:
# --- Test 3: Tech support ticket routing ---
# Imagine an IT helpdesk — classify incoming tickets with zero labelled data

tickets = [
    "My laptop won't turn on after the Windows update last night.",
    "I need to reset my password for the company email account.",
    "The conference room projector cable is missing again.",
    "Our team needs 5 new software licenses for Adobe Acrobat."
]
it_categories = ["hardware issue", "account and access", "equipment request", "software licensing"]

print("IT Helpdesk Ticket Routing (zero-shot):")
print("=" * 70)
for ticket in tickets:
    result = zeroshot_pipeline(ticket, it_categories)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    print(f"Ticket:   {ticket}")
    print(f"Route to: [{top_label.upper()}]  (confidence: {top_score:.3f})")
    print()

In [ ]:
# --- Test 4: Multi-label classification ---
# A piece of text can belong to multiple categories simultaneously

text_multilabel = (
    "The Nepal government announced a new policy to attract foreign investment "
    "in the renewable energy sector, particularly hydropower projects."
)
labels = ["government policy", "economics", "environment", "energy", "international relations"]

result = zeroshot_pipeline(text_multilabel, labels, multi_label=True)

print(f"Text: {text_multilabel}")
print(f"Multi-label=True (scores are independent, can all be high)")
print()
print(f"{'Label':<25} {'Score':<10}")
print("-" * 38)
for label, score in zip(result['labels'], result['scores']):
    bar = '█' * int(score * 25)
    print(f"{label:<25} {score:.3f}  {bar}")

**Key observations:**
- Zero labelled examples needed — labels defined at runtime
- Works by framing classification as natural language entailment
- `multi_label=True` allows multiple labels to score high simultaneously
- Extremely useful for rapid prototyping before you've collected a labelled dataset
- Demonstrates the power of representations learned from pre-training on massive text corpora

---

## Demo 6 — Bonus: See What Model Is Running

Students often wonder: *what model is actually inside the pipeline?* Let's inspect.

In [ ]:
# Inspect model details behind each pipeline
pipelines = {
    "sentiment-analysis":        sentiment_pipeline,
    "ner":                        ner_pipeline,
    "question-answering":         qa_pipeline,
    "summarization":              summarizer,
    "zero-shot-classification":   zeroshot_pipeline,
}

print(f"{'Task':<30} {'Model Name':<45} {'Params (approx)':<15}")
print("-" * 95)

for task, pipe in pipelines.items():
    model_name = pipe.model.config._name_or_path
    # Count parameters
    num_params = sum(p.numel() for p in pipe.model.parameters())
    param_str = f"{num_params / 1e6:.0f}M"
    print(f"{task:<30} {model_name:<45} {param_str:<15}")

In [ ]:
# What architecture family is each model?
print("Architecture families:")
print("-" * 50)
for task, pipe in pipelines.items():
    arch = pipe.model.config.architectures
    print(f"{task:<30} {arch}")

---

## Summary

| Pipeline | Task Type | Model Family | Architecture |
|----------|-----------|--------------|---------------|
| `sentiment-analysis` | Sequence Classification | DistilBERT | Encoder only |
| `ner` | Token Classification | BERT | Encoder only |
| `question-answering` | Span Extraction | RoBERTa | Encoder only |
| `summarization` | Text Generation | BART | Encoder + Decoder |
| `zero-shot-classification` | NLI-based Classification | BART | Encoder + Decoder |

**Pattern:**
- **Understanding tasks** (classify, extract, find) → Encoder-only models (BERT family)
- **Generation tasks** (summarize, translate, generate) → Encoder-Decoder or Decoder-only models

This directly maps to everything we covered in today's session.

---

## What's Next — Day 14

Tomorrow you write this code yourself. You will:
1. Load a pretrained BERT tokenizer and understand how text becomes token IDs
2. Fine-tune a BERT model on a custom classification dataset
3. Build an end-to-end NLP pipeline from raw text to prediction
4. Explore model outputs: logits, probabilities, attention weights

Everything you saw today becomes code you own tomorrow.

---

> *"Attention Is All You Need" — Vaswani et al., 2017*